# Construcción del dataset gold de 1.400 publicaciones

Este cuaderno reúne la construcción del conjunto de evaluación humana a partir de `bdi_total.parquet`. El resultado principal es `datos/gold_1400.csv`, con 1.400 publicaciones estratificadas por subreddit y banda de `bdi_prob`.

El trabajo está dividido en cinco secciones:

1. Muestreo estratificado y escritura de `datos/gold_1400.csv`.
2. Preetiquetado binario con GLM y actualización de las columnas `api`.
3. Preparación del corpus para el MLM sin las publicaciones del gold.
4. Revisión humana binaria (`revisar.py`) y análisis de concordancia.
5. Anotación de los 21 síntomas del BDI-II sobre los positivos (`revisar_sintomas.py`).

Abre el cuaderno desde `02_dataset_gold` para que las rutas relativas apunten a `01_datos_reddit`, `02_dataset_gold` y `03_adaptacion_mlm`. Los archivos existentes se conservan salvo que actives la sobrescritura correspondiente.


## Configuración

Rutas, parámetros del muestreo y banderas de protección (`SOBRESCRIBIR_GOLD` y `SOBRESCRIBIR_MLM`).


In [30]:
import hashlib
import json
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from tqdm.auto import tqdm


In [31]:
BDI_TOTAL = Path("../01_datos_reddit/datos/bdi_total.parquet")
DATOS_DIR = Path("datos")
ARCHIVO_GOLD = DATOS_DIR / "gold_1400.csv"
ESTRATOS_JSON = DATOS_DIR / "estratos.json"
PREETIQUETAS_JSONL = DATOS_DIR / "preetiquetas.jsonl"
MLM_CORPUS_OUT = Path("../03_adaptacion_mlm/artefactos/mlm_corpus.parquet")
PREETIQUETAS_BDI21_JSONL = DATOS_DIR / "preetiquetas_bdi21.jsonl"
ARCHIVO_BDI21 = DATOS_DIR / "gold_bdi21.csv"
SOBRESCRIBIR_GOLD = False
SOBRESCRIBIR_MLM = False

SUBREDDITS = [
    "asktransgender",
    "ftm",
    "mtf",
    "nonbinary",
    "trans",
    "translater",
    "transmasc",
]

BANDAS = [
    {"lo": 0.00, "hi": 0.10, "frac": 0.30, "nombre": "0.00-0.10"},
    {"lo": 0.10, "hi": 0.45, "frac": 0.20, "nombre": "0.10-0.45"},
    {"lo": 0.45, "hi": 0.65, "frac": 0.30, "nombre": "0.45-0.65"},
    {"lo": 0.65, "hi": 0.90, "frac": 0.10, "nombre": "0.65-0.90"},
    {"lo": 0.90, "hi": 1.01, "frac": 0.10, "nombre": "0.90-1.00"},
]

CUOTA_POR_SUBREDDIT = 200
MIN_PALABRAS = 20
SEED = 20261122

COLUMNAS_GOLD = [
    "orden",
    "id",
    "subreddit",
    "anio",
    "palabras",
    "bdi_prob",
    "banda",
    "gold",
    "nota",
    "titulo",
    "texto",
    "author_hash",
    "api",
    "api_motivo",
]

COLUMNAS_MLM = ["id", "subreddit", "created_year", "title", "text", "bdi_prob"]

AUTORES_GENERICOS = {
    "[deleted]",
    "[removed]",
    "deleted",
    "removed",
    "AutoModerator",
    "automoderator",
    "None",
    "nan",
    "",
}


## 1. Muestreo estratificado

Se seleccionan 200 publicaciones por subreddit: 60 de la banda `0.00-0.10`, 40 de `0.10-0.45`, 60 de `0.45-0.65`, 20 de `0.65-0.90` y 20 de `0.90-1.00`. La semilla es fija y el muestreo se hace por estrato, así que repetirlo produce los mismos 1.400 identificadores.

Si `datos/gold_1400.csv` ya existe y `SOBRESCRIBIR_GOLD` es `False`, el archivo se carga sin modificarse. `datos/estratos.json` guarda el tamaño poblacional de cada estrato y se usa después para reponderar métricas.


In [32]:
DATOS_DIR.mkdir(parents=True, exist_ok=True)

rehacer_gold = SOBRESCRIBIR_GOLD or not ARCHIVO_GOLD.exists()
necesita_estratos = SOBRESCRIBIR_GOLD or not ESTRATOS_JSON.exists()

if rehacer_gold or necesita_estratos:
    if not BDI_TOTAL.exists():
        raise FileNotFoundError(f"No existe el corpus: {BDI_TOTAL.resolve()}")

    pool = pd.read_parquet(
        BDI_TOTAL,
        columns=["id", "author", "subreddit", "created_year", "title", "text", "bdi_prob"],
    )
    pool["subreddit"] = pool["subreddit"].fillna("").astype(str).str.strip().str.lower()
    pool = pool[pool["subreddit"].isin(SUBREDDITS)].copy()
    pool["titulo"] = pool["title"].fillna("").astype(str).str.strip()
    pool["cuerpo"] = pool["text"].fillna("").astype(str).str.strip()
    pool["palabras"] = (pool["titulo"] + " " + pool["cuerpo"]).str.count(r"\S+")
    pool = pool[
        pool["bdi_prob"].notna()
        & (pool["palabras"] >= MIN_PALABRAS)
        & (pool["cuerpo"] != "")
    ].copy()

    pool["banda"] = ""
    for banda in BANDAS:
        dentro = (pool["bdi_prob"] >= banda["lo"]) & (pool["bdi_prob"] < banda["hi"])
        pool.loc[dentro, "banda"] = banda["nombre"]
    pool.loc[pool["banda"] == "", "banda"] = BANDAS[-1]["nombre"]

    conteos = pool.groupby(["subreddit", "banda"]).size()

    if necesita_estratos:
        estratos = {
            f"{sub}|{banda['nombre']}": int(conteos.get((sub, banda["nombre"]), 0))
            for sub in SUBREDDITS
            for banda in BANDAS
        }
        parametros = {
            "por_subreddit": CUOTA_POR_SUBREDDIT,
            "min_palabras": MIN_PALABRAS,
            "seed": SEED,
            "origen": str(BDI_TOTAL),
            "bandas": BANDAS,
        }
        ESTRATOS_JSON.write_text(
            json.dumps(
                {"estratos": estratos, "parametros": parametros},
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"estratos.json: {len(estratos)} estratos")

    if rehacer_gold:
        grupos = {}
        for indice_sub, sub in enumerate(SUBREDDITS):
            grupo = pool[pool["subreddit"] == sub]
            tomados = []
            for indice_banda, banda in enumerate(BANDAS):
                objetivo = int(round(CUOTA_POR_SUBREDDIT * banda["frac"]))
                disponibles = grupo[grupo["banda"] == banda["nombre"]]
                rng = np.random.default_rng([SEED, indice_sub, indice_banda])
                if len(disponibles) >= objetivo:
                    tomados.append(disponibles.sample(n=objetivo, random_state=rng))
                else:
                    print(
                        f"{sub} | {banda['nombre']}: hay {len(disponibles)} de "
                        f"{objetivo}; se toman todas"
                    )
                    tomados.append(disponibles)

            parcial = pd.concat(tomados)
            if len(parcial) < CUOTA_POR_SUBREDDIT:
                resto = grupo.drop(index=parcial.index)
                rng = np.random.default_rng([SEED, indice_sub, 98])
                falta = min(CUOTA_POR_SUBREDDIT - len(parcial), len(resto))
                if falta:
                    print(f"{sub}: se rellenan {falta} publicaciones de otras bandas")
                    parcial = pd.concat(
                        [parcial, resto.sample(n=falta, random_state=rng)]
                    )

            rng = np.random.default_rng([SEED, indice_sub, 99])
            parcial = parcial.sample(frac=1.0, random_state=rng).reset_index(drop=True)
            parcial["orden"] = np.arange(1, len(parcial) + 1)
            grupos[sub] = parcial

        muestra = pd.concat([grupos[sub] for sub in SUBREDDITS], ignore_index=True)
        muestra["anio"] = muestra["created_year"]
        muestra["texto"] = muestra["cuerpo"]
        muestra["gold"] = ""
        muestra["nota"] = ""
        muestra["api"] = ""
        muestra["api_motivo"] = ""
        muestra["author_hash"] = muestra["author"].map(
            lambda autor: hashlib.sha256(str(autor).encode("utf-8")).hexdigest()[:12]
            if isinstance(autor, str) and autor
            else None
        )
        muestra[COLUMNAS_GOLD].to_csv(ARCHIVO_GOLD, index=False, encoding="utf-8-sig")
        print(f"gold_1400.csv: {len(muestra):,} filas")
        del grupos

    del pool

if not ARCHIVO_GOLD.exists():
    raise FileNotFoundError(
        f"No existe {ARCHIVO_GOLD} y SOBRESCRIBIR_GOLD es False; "
        "activa la sobrescritura para generarlo."
    )

gold = pd.read_csv(ARCHIVO_GOLD, encoding="utf-8-sig", dtype={"id": str})
gold["id"] = gold["id"].astype(str)
faltan = [columna for columna in COLUMNAS_GOLD if columna not in gold.columns]
if faltan:
    raise ValueError(f"Faltan columnas en {ARCHIVO_GOLD}: {faltan}")
gold = gold[COLUMNAS_GOLD]

esperado = CUOTA_POR_SUBREDDIT * len(SUBREDDITS)
if len(gold) != esperado:
    print(f"Aviso: se esperaban {esperado:,} filas y hay {len(gold):,}.")

print()
print(f"gold: {len(gold):,} filas · {gold['subreddit'].nunique()} subreddits")
print(gold.groupby("subreddit").size().to_string())
print(f"con api: {int(pd.to_numeric(gold['api'], errors='coerce').notna().sum()):,}")
print(
    "gold humano 0/1: "
    f"{int(pd.to_numeric(gold['gold'], errors='coerce').isin([0, 1]).sum()):,}"
)



gold: 1,400 filas · 7 subreddits
subreddit
asktransgender    200
ftm               200
mtf               200
nonbinary         200
trans             200
translater        200
transmasc         200
con api: 1,400
gold humano 0/1: 1,400


## 2. Preetiquetado binario con GLM

Cada llamada envía diez publicaciones al modelo `zai-glm-5-3` de Mistral. Se usan cuatro hilos, reintentos con espera exponencial y una caché JSONL en `datos/preetiquetas.jsonl`. La clave se lee de la variable de entorno `MISTRAL_API_KEY`; nunca se guarda en el cuaderno.

Para pausar, interrumpe la celda: lo ya escrito en la caché se conserva y al volver a ejecutarla solo se piden las publicaciones pendientes. Cuando los 1.400 identificadores ya están en caché, no hace falta definir la clave. Las columnas humanas `gold` y `nota` no se tocan.


In [33]:
BASE_URL = "https://api.mistral.ai/v1"
MODELO_GLM = "zai-glm-5-3"
TANDA = 10
TRABAJADORES = 4
MAX_POSTS = None
TEMPERATURA = 0.0
CLAVE = os.environ.get("MISTRAL_API_KEY", "").strip()

SISTEMA_GLM = (
    "You label Reddit posts for a research dataset on depression. "
    "For each post decide if the AUTHOR describes that they themselves experience at least one "
    "depression symptom: sadness, hopelessness, feeling a failure, loss of pleasure or interest, "
    "guilt, self-dislike, worthlessness, suicidal thoughts, crying, restlessness, indecisiveness, "
    "low energy, sleep or appetite changes, irritability, poor concentration, or fatigue. "
    "Rules: gender dysphoria alone is NOT a symptom; depression mentioned about other people, "
    "quoted, or as a joke does NOT count; a question about the topic does NOT count unless the "
    "author includes themselves. "
    "Answer with ONLY a JSON object, no prose: "
    '{"labels":[{"i":1,"y":1,"w":"short reason"}]} '
    "where i is the post number, y is 1 or 0, and w is at most 10 words."
)


def cargar_jsonl(ruta):
    if not ruta.exists():
        return []
    filas = []
    with open(ruta, encoding="utf-8") as archivo:
        for linea in archivo:
            linea = linea.strip()
            if not linea:
                continue
            try:
                dato = json.loads(linea)
            except json.JSONDecodeError:
                continue
            if isinstance(dato, dict):
                filas.append(dato)
    return filas


def armar_usuario(posts):
    bloques = [
        f"[{numero}] TITLE: {post['titulo']}\nBODY: {post['texto']}"
        for numero, post in enumerate(posts, start=1)
    ]
    return "Label these posts.\n\n" + "\n\n---\n\n".join(bloques)


def texto_de(mensaje):
    contenido = mensaje.get("content", "")
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        return "".join(
            parte.get("text", "")
            for parte in contenido
            if isinstance(parte, dict) and parte.get("type") == "text"
        )
    return ""


def parsear_lote(texto, posts):
    if not isinstance(texto, str):
        return {}
    dato = None
    for inicio, fin in (
        (0, len(texto) - 1),
        (texto.find("{"), texto.rfind("}")),
        (texto.find("["), texto.rfind("]")),
    ):
        if inicio < 0 or fin <= inicio:
            continue
        try:
            dato = json.loads(texto[inicio:fin + 1])
            break
        except json.JSONDecodeError:
            continue
    if dato is None:
        return {}
    if isinstance(dato, list):
        dato = {"labels": dato}
    if not isinstance(dato, dict):
        return {}
    items = dato.get("labels", dato)
    if isinstance(items, dict):
        items = [{"i": clave, "y": valor} for clave, valor in items.items()]
    if not isinstance(items, list):
        return {}
    salida = {}
    for item in items:
        try:
            indice = int(item["i"]) - 1
            if 0 <= indice < len(posts):
                y = int(item["y"])
                if y in (0, 1):
                    salida[posts[indice]["id"]] = (y, str(item.get("w", ""))[:140])
        except (KeyError, TypeError, ValueError):
            continue
    return salida


def etiquetar_tanda(posts, reintentos=4, tope=4800):
    if not CLAVE:
        raise RuntimeError(
            "Falta MISTRAL_API_KEY en las variables de entorno para consultar GLM."
        )
    cabeceras = {
        "Authorization": f"Bearer {CLAVE}",
        "Content-Type": "application/json",
    }
    carga = {
        "model": MODELO_GLM,
        "temperature": TEMPERATURA,
        "max_tokens": tope,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SISTEMA_GLM},
            {"role": "user", "content": armar_usuario(posts)},
        ],
    }
    for intento in range(reintentos):
        try:
            respuesta = requests.post(
                f"{BASE_URL}/chat/completions",
                headers=cabeceras,
                json=carga,
                timeout=120,
            )
        except requests.RequestException as exc:
            print(f"error de red ({exc}); se reintenta")
            time.sleep(min(2 ** intento, 15))
            continue

        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            espera = min(2 ** intento + random.random(), 20)
            print(f"http {respuesta.status_code}; se espera {espera:.1f}s")
            time.sleep(espera)
            continue
        if respuesta.status_code != 200:
            print(f"http {respuesta.status_code}: {respuesta.text[:250]}")
            return {}

        motivo_fin = None
        salida = {}
        try:
            dato = respuesta.json()
            eleccion = dato["choices"][0]
            motivo_fin = eleccion.get("finish_reason")
            salida = parsear_lote(texto_de(eleccion["message"]), posts)
        except (KeyError, IndexError, TypeError, ValueError) as exc:
            print(f"respuesta no legible ({exc}); se reintenta")
            time.sleep(min(2 ** intento, 15))
            continue

        if len(salida) == len(posts):
            return salida
        if motivo_fin == "length" and tope < 12000:
            nuevo_tope = min(tope * 2, 12000)
            print(f"respuesta cortada con {tope} tokens ({len(salida)}/{len(posts)}); subo a {nuevo_tope}")
            return etiquetar_tanda(posts, reintentos=max(1, reintentos - 1), tope=nuevo_tope)
        if salida:
            return salida
        time.sleep(min(2 ** intento, 10))
    return {}


In [34]:
hechos = {
    str(r.get("id", "")).strip()
    for r in cargar_jsonl(PREETIQUETAS_JSONL)
    if r.get("y") in (0, 1)
}
cola = gold.head(MAX_POSTS) if MAX_POSTS else gold
pendientes = [
    fila
    for fila in cola[["id", "titulo", "texto"]].fillna("").to_dict("records")
    if str(fila["id"]) not in hechos
]
print(f"en caché: {len(hechos):,} · pendientes: {len(pendientes):,}")

if pendientes:
    tandas = [pendientes[i:i + TANDA] for i in range(0, len(pendientes), TANDA)]
    print(f"lanzando {len(tandas)} tandas con {TRABAJADORES} hilos...")
    fallidas = 0
    try:
        with (
            ThreadPoolExecutor(max_workers=TRABAJADORES) as ejecutor,
            open(PREETIQUETAS_JSONL, "a", encoding="utf-8") as salida,
            tqdm(total=len(tandas), unit="tanda", desc="GLM") as barra,
        ):
            futuros = {ejecutor.submit(etiquetar_tanda, t): t for t in tandas}
            for futuro in as_completed(futuros):
                tanda_posts = futuros[futuro]
                try:
                    res = futuro.result()
                except Exception as exc:
                    print("error en tanda:", exc)
                    res = {}
                if not res:
                    fallidas += 1
                for post_id, (y, motivo) in res.items():
                    salida.write(
                        json.dumps(
                            {"id": post_id, "y": int(y), "w": motivo, "modelo": MODELO_GLM},
                            ensure_ascii=False,
                        )
                        + "\n"
                    )
                salida.flush()
                barra.update(1)
    except KeyboardInterrupt:
        print("\nPausado por el usuario; el progreso en JSONL quedó guardado.")
    print(f"tandas sin resultado: {fallidas}")


en caché: 1,400 · pendientes: 0


In [35]:
cache_api = {}
for r in cargar_jsonl(PREETIQUETAS_JSONL):
    pid = str(r.get("id", "")).strip()
    if pid and r.get("y") in (0, 1):
        cache_api[pid] = r

if cache_api:
    gold_actualizado = gold.copy()
    api_previa = pd.to_numeric(gold_actualizado["api"], errors="coerce")
    api_nueva = gold_actualizado["id"].map(
        lambda x: cache_api[x]["y"] if x in cache_api else np.nan
    )
    api_final = api_nueva.where(api_nueva.notna(), api_previa).astype("Int64")

    motivo_previo = gold_actualizado["api_motivo"].fillna("").astype(str)
    motivo_nuevo = gold_actualizado["id"].map(
        lambda x: cache_api[x].get("w", "") if x in cache_api else None
    )
    gold_actualizado["api"] = api_final
    gold_actualizado["api_motivo"] = motivo_nuevo.where(motivo_nuevo.notna(), motivo_previo)

    if not gold_actualizado.equals(gold):
        temporal = ARCHIVO_GOLD.with_suffix(".tmp")
        gold_actualizado.to_csv(temporal, index=False, encoding="utf-8-sig")
        os.replace(temporal, ARCHIVO_GOLD)
        gold = gold_actualizado
        print("Actualizado datos/gold_1400.csv conservando gold y nota.")
    else:
        print("datos/gold_1400.csv ya estaba sincronizado con la caché.")

api_num = pd.to_numeric(gold["api"], errors="coerce")
print(f"etiquetas api válidas: {int(api_num.isin([0, 1]).sum()):,}/{len(gold):,}")
if api_num.isin([0, 1]).any():
    print(f"positivos GLM: {int((api_num == 1).sum()):,} ({100 * (api_num == 1).mean():.2f}%)")
    print(gold.assign(api_pos=(api_num == 1).astype(float)).groupby("subreddit")["api_pos"].mean().round(3).to_string())


Actualizado datos/gold_1400.csv conservando gold y nota.
etiquetas api válidas: 1,400/1,400
positivos GLM: 191 (13.64%)
subreddit
asktransgender    0.155
ftm               0.135
mtf               0.145
nonbinary         0.110
trans             0.160
translater        0.130
transmasc         0.120


## 3. Corpus para el MLM sin fuga del gold

Excluye los 1.400 identificadores de `datos/gold_1400.csv` de `bdi_total.parquet` y guarda `../03_adaptacion_mlm/artefactos/mlm_corpus.parquet` si no existe o si `SOBRESCRIBIR_MLM` es `True`. También cuenta cuántas publicaciones del corpus corresponden a los autores del gold (esa exclusión por autor se aplica luego en el notebook 03).


In [36]:
ids_gold = set(gold["id"].astype(str))
MLM_CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

if SOBRESCRIBIR_MLM or not MLM_CORPUS_OUT.exists():
    if not BDI_TOTAL.exists():
        raise FileNotFoundError(f"No existe {BDI_TOTAL}")
    tabla = pq.read_table(BDI_TOTAL, columns=COLUMNAS_MLM).to_pandas()
    tabla["id"] = tabla["id"].astype(str)
    sin_gold = tabla[~tabla["id"].isin(ids_gold)].copy()
    del tabla
    pq.write_table(
        pa.Table.from_pandas(sin_gold, preserve_index=False),
        MLM_CORPUS_OUT,
        compression="zstd",
    )
    print(f"Escrito {MLM_CORPUS_OUT} ({MLM_CORPUS_OUT.stat().st_size / 1e6:.1f} MB)")
    del sin_gold
else:
    print(f"Conservado {MLM_CORPUS_OUT} (SOBRESCRIBIR_MLM=False)")

ids_mlm = set(pq.read_table(MLM_CORPUS_OUT, columns=["id"])["id"].to_pylist())
ids_mlm = {str(x) for x in ids_mlm}
fuga = len(ids_gold & ids_mlm)
assert fuga == 0, f"Fuga detectada: {fuga} IDs del gold dentro de mlm_corpus.parquet"
print(f"Verificación anti-fuga: {len(ids_mlm):,} filas en mlm_corpus · 0 IDs del gold dentro")

if BDI_TOTAL.exists():
    au = pq.read_table(BDI_TOTAL, columns=["id", "author"]).to_pandas()
    au["id"] = au["id"].astype(str)
    autores_gold = set(au.loc[au["id"].isin(ids_gold), "author"].dropna().astype(str))
    autores_gold -= AUTORES_GENERICOS
    ids_de_autores = set(au.loc[au["author"].isin(autores_gold), "id"].astype(str))
    en_mlm_autores = len(ids_de_autores & ids_mlm)
    print(f"Autores del gold: {len(autores_gold):,} · posts suyos aún en mlm_corpus: {en_mlm_autores:,}")
    del au


Conservado ../03_adaptacion_mlm/artefactos/mlm_corpus.parquet (SOBRESCRIBIR_MLM=False)
Verificación anti-fuga: 1,547,536 filas en mlm_corpus · 0 IDs del gold dentro
Autores del gold: 1,347 · posts suyos aún en mlm_corpus: 23,141


## 4. Revisión humana binaria y análisis de concordancia

La revisión manual de las 1.400 publicaciones se hace en terminal desde `02_dataset_gold/`:

```bash
cd "02_dataset_gold"
py revisar.py        # en WSL: python3 revisar.py
```

Cada pulsación guarda de inmediato en `datos/gold_1400.csv` y registra la bitácora en `datos/revision.jsonl`. La siguiente celda resume el avance, compara `bdi_prob >= 0.5` frente a `api` (funciona incluso antes de empezar la revisión humana) y, cuando ya haya etiquetas humanas `gold` (`0`/`1`), calcula concordancia y métricas reponderadas usando los tamaños poblacionales de `datos/estratos.json`.


In [37]:
def calcular_kappa(y1, y2, pesos=None):
    y1 = np.asarray(y1, dtype=int)
    y2 = np.asarray(y2, dtype=int)
    if len(y1) == 0:
        return float("nan")
    w = np.ones(len(y1), dtype=float) if pesos is None else np.asarray(pesos, dtype=float)
    w = w / w.sum()
    po = float(w[y1 == y2].sum())
    p1_a, p1_b = float(w[y1 == 1].sum()), float(w[y2 == 1].sum())
    pe = p1_a * p1_b + (1.0 - p1_a) * (1.0 - p1_b)
    return float((po - pe) / (1.0 - pe)) if pe < 1.0 else 1.0


def calcular_metricas(y_real, y_pred, pesos=None):
    y_real = np.asarray(y_real, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if len(y_real) == 0:
        return None
    w = np.ones(len(y_real), dtype=float) if pesos is None else np.asarray(pesos, dtype=float)
    w = w / w.sum()
    tp = float(w[(y_real == 1) & (y_pred == 1)].sum())
    fp = float(w[(y_real == 0) & (y_pred == 1)].sum())
    fn = float(w[(y_real == 1) & (y_pred == 0)].sum())
    tn = float(w[(y_real == 0) & (y_pred == 0)].sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    rec = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float("nan")
    return {
        "n": len(y_real),
        "prevalencia_real": tp + fn,
        "acc": tp + tn,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "kappa": calcular_kappa(y_real, y_pred, pesos=w),
    }


t = pd.read_csv(ARCHIVO_GOLD, encoding="utf-8-sig", dtype={"id": str})
t["subreddit"] = t["subreddit"].fillna("").astype(str).str.strip().str.lower()
t["banda"] = t["banda"].fillna("").astype(str).str.strip()
humano = pd.to_numeric(t["gold"], errors="coerce")
api_v = pd.to_numeric(t["api"], errors="coerce")
bdi_v = pd.to_numeric(t["bdi_prob"], errors="coerce")

t["humano"] = humano.where(humano.isin([0, 1]))
t["dudoso"] = humano.eq(9)
t["api_num"] = api_v.where(api_v.isin([0, 1]))
t["bdi_pred"] = (bdi_v >= 0.5).where(bdi_v.notna()).astype("Int64")

avance = (
    t.assign(
        hecho_01=t["humano"].notna(),
        pos_1=t["humano"].eq(1),
        neg_0=t["humano"].eq(0),
        dud=t["dudoso"],
    )
    .groupby("subreddit")
    .agg(total=("id", "size"), hechos=("hecho_01", "sum"), pos=("pos_1", "sum"), neg=("neg_0", "sum"), dudosos=("dud", "sum"))
)
print("Avance de revisión humana por subreddit:")
print(avance.to_string())

pares_bdi = t[["bdi_pred", "api_num"]].dropna().astype(int)
print("\nTabla cruzada bdi_prob >= 0.5 (filas) vs GLM api (columnas):")
print(pd.crosstab(pares_bdi["bdi_pred"], pares_bdi["api_num"], margins=True).to_string())
print(f"Acuerdo bruto: {100 * (pares_bdi['bdi_pred'] == pares_bdi['api_num']).mean():.2f}% · "
      f"kappa={calcular_kappa(pares_bdi['bdi_pred'], pares_bdi['api_num']):.3f}")

validos = t["humano"].notna()
if validos.any():
    poblacion = json.loads(ESTRATOS_JSON.read_text(encoding="utf-8"))["estratos"] if ESTRATOS_JSON.exists() else {}
    clave = t["subreddit"] + "|" + t["banda"]
    n_estrato = clave.value_counts().to_dict()
    peso = clave.map(lambda k: poblacion[k] / n_estrato[k] if k in poblacion else 1.0)

    print(f"\nMétricas humanas disponibles (n={int(validos.sum()):,}):")
    for nombre, col in (("GLM (api)", "api_num"), ("Detector previo (bdi_prob>=0.5)", "bdi_pred")):
        p = t.loc[validos, ["humano", col]].dropna()
        if p.empty:
            continue
        m_crudo = calcular_metricas(p["humano"].astype(int), p[col].astype(int))
        m_pond = calcular_metricas(p["humano"].astype(int), p[col].astype(int), pesos=peso.loc[p.index])
        print(f"\n[{nombre}]")
        for tipo, m in (("Muestra (sin reponderar)", m_crudo), ("Población (estratos.json)", m_pond)):
            print(
                f"  {tipo:<26} n={m['n']:>4} · prev={100*m['prevalencia_real']:.1f}% · "
                f"acc={100*m['acc']:.1f}% · P={100*m['precision']:.1f}% · "
                f"R={100*m['recall']:.1f}% · F1={100*m['f1']:.1f}% · kappa={m['kappa']:.3f}"
            )
else:
    print("\nAún no hay etiquetas humanas en la columna `gold`. Ejecuta `revisar.py` cuando estés lista.")


Avance de revisión humana por subreddit:
                total  hechos  pos  neg  dudosos
subreddit                                       
asktransgender    200     200   29  171        0
ftm               200     200   24  176        0
mtf               200     200   28  172        0
nonbinary         200     200   21  179        0
trans             200     200   30  170        0
translater        200     200   25  175        0
transmasc         200     200   23  177        0

Tabla cruzada bdi_prob >= 0.5 (filas) vs GLM api (columnas):
api_num      0    1   All
bdi_pred                 
0          799    5   804
1          410  186   596
All       1209  191  1400
Acuerdo bruto: 70.36% · kappa=0.335

Métricas humanas disponibles (n=1,400):

[GLM (api)]
  Muestra (sin reponderar)   n=1400 · prev=12.9% · acc=99.1% · P=93.7% · R=99.4% · F1=96.5% · kappa=0.960
  Población (estratos.json)  n=1400 · prev=16.1% · acc=99.1% · P=95.9% · R=99.0% · F1=97.4% · kappa=0.969

[Detector previo (bdi_p

## 5. Subconjunto de los 21 síntomas del BDI-II sobre los positivos

Toma todas las publicaciones confirmadas como positivas (`gold == 1`) en `datos/gold_1400.csv` y preetiqueta los 21 síntomas canónicos del BDI-II (en el mismo orden `1..21` que `eRisk_2024_dataset_majority.csv`).

Para preetiquetar, ejecuta **solo una** de las dos celdas siguientes (ambas escriben en la misma caché reanudable `datos/preetiquetas_bdi21.jsonl`):
- **Opción A (GLM · Mistral API):** usa `zai-glm-5-3` vía `/v1/chat/completions` con `MISTRAL_API_KEY`.
- **Opción B (StartLux local · `/v1/systemone`):** usa tu servidor local (`127.0.0.1:8889`) con preguntas de tipo `noul` (una por cada uno de los 21 síntomas en una sola petición por post).

Después corre la celda de armado de `datos/gold_bdi21.csv` (siempre conserva intactas las filas ya revisadas manualmente).


In [41]:
import http.client
import threading

TRABAJADORES_SINTOMAS = 4

BDI_21 = [
    (1, "Sadness", "Tristeza"),
    (2, "Pessimism", "Pesimismo"),
    (3, "Past Failure", "Fracaso pasado"),
    (4, "Loss of Pleasure", "Perdida de placer"),
    (5, "Guilty Feelings", "Sentimientos de culpa"),
    (6, "Punishment Feelings", "Sentimientos de castigo"),
    (7, "Self-Dislike", "Rechazo hacia si mismo"),
    (8, "Self-Criticalness", "Autocritica"),
    (9, "Suicidal Thoughts or Wishes", "Pensamientos suicidas"),
    (10, "Crying", "Llanto"),
    (11, "Agitation", "Agitacion"),
    (12, "Loss of Interest", "Perdida de interes"),
    (13, "Indecisiveness", "Indecision"),
    (14, "Worthlessness", "Inutilidad o falta de valor"),
    (15, "Loss of Energy", "Perdida de energia"),
    (16, "Changes in Sleeping Pattern", "Cambios en el sueno"),
    (17, "Irritability", "Irritabilidad"),
    (18, "Changes in Appetite", "Cambios en el apetito"),
    (19, "Concentration Difficulty", "Dificultad de concentracion"),
    (20, "Tiredness or Fatigue", "Cansancio o fatiga"),
    (21, "Loss of Interest in Sex", "Perdida de interes en el sexo"),
]
CAMPOS_S = [f"s{n:02d}" for n in range(1, 22)]
COLUMNAS_BDI21 = [
    "orden", "id", "subreddit", "anio", "palabras", "bdi_prob",
    "titulo", "texto", "pre_modelo", "pre_sintomas", "pre_motivo",
    "gold_sintomas", "revisado",
] + CAMPOS_S

gold_actual = pd.read_csv(ARCHIVO_GOLD, encoding="utf-8-sig", dtype={"id": str})
positivos = gold_actual[pd.to_numeric(gold_actual["gold"], errors="coerce") == 1].copy()
print(f"Positivos confirmados (gold == 1): {len(positivos):,}")
if positivos.empty:
    print("Todavía no hay positivos humanos en datos/gold_1400.csv.")
    print("Termina la revisión binaria con `revisar.py` y vuelve a correr esta sección.")


def ejecutar_preetiquetado_bdi21(fn, nombre_modelo, trabajadores=TRABAJADORES_SINTOMAS):
    if positivos.empty:
        print("Sin positivos confirmados; no se lanzó ninguna llamada.")
        return
    cache_b21 = {
        str(r.get("id", "")).strip(): r
        for r in cargar_jsonl(PREETIQUETAS_BDI21_JSONL)
        if isinstance(r.get("bits"), list) and len(r["bits"]) == 21
    }
    pend_b21 = positivos[~positivos["id"].isin(set(cache_b21))].copy()
    print(f"[{nombre_modelo}] en caché BDI-21: {len(cache_b21):,} · pendientes: {len(pend_b21):,}")
    if pend_b21.empty:
        return

    regs = pend_b21[["id", "titulo", "texto"]].fillna("").to_dict("records")
    fallos = 0
    try:
        with (
            ThreadPoolExecutor(max_workers=min(trabajadores, len(regs))) as ejecutor,
            open(PREETIQUETAS_BDI21_JSONL, "a", encoding="utf-8") as salida,
            tqdm(total=len(regs), unit="post", desc=f"BDI-21 ({nombre_modelo})") as barra,
        ):
            futuros = {ejecutor.submit(fn, r["titulo"], r["texto"]): r["id"] for r in regs}
            for futuro in as_completed(futuros):
                pid = futuros[futuro]
                try:
                    res = futuro.result()
                except Exception as exc:
                    print(f"error en {pid}: {exc}")
                    res = None
                if res is None:
                    fallos += 1
                else:
                    bits, motivo, extras = res if len(res) == 3 else (*res, {})
                    activos = ",".join(str(i + 1) for i, b in enumerate(bits) if b == 1)
                    reg = {
                        "id": pid,
                        "modelo": nombre_modelo,
                        "bits": bits,
                        "pre_sintomas": activos,
                        "pre_motivo": motivo,
                    }
                    if extras:
                        reg.update(extras)
                    cache_b21[pid] = reg
                    salida.write(json.dumps(reg, ensure_ascii=False) + "\n")
                    salida.flush()
                barra.update(1)
    except KeyboardInterrupt:
        print("\nPausado por el usuario; caché BDI-21 guardada.")
    print(f"[{nombre_modelo}] posts sin preetiqueta: {fallos}")


Positivos confirmados (gold == 1): 180


### Opción A · Preetiquetar 21 síntomas con GLM (`zai-glm-5-3`, Mistral API)

Requiere la variable de entorno `MISTRAL_API_KEY`. Si prefieres usar tu servidor local StartLux, sáltate esta celda y corre la **Opción B**.


In [42]:
LISTA_SINTOMAS = "\n".join(f"{n}. {en} ({es})" for n, en, es in BDI_21)

SISTEMA_BDI21 = (
    "You are a clinical NLP annotator. For a single Reddit post by a depressed author, "
    "decide which of the 21 BDI-II symptoms the author explicitly describes experiencing in first person. "
    "Mark 1 only if present in the post; 0 otherwise.\n"
    f"Symptoms (1..21 in exact order):\n{LISTA_SINTOMAS}\n"
    'Return ONLY valid JSON: {"sintomas":[0,...,0],"motivo":"short 10-word summary"} '
    "where sintomas is an array of exactly 21 integers (0 or 1)."
)


def parsear_sintomas_21(texto):
    if not isinstance(texto, str):
        return None
    dato = None
    for inicio, fin in (
        (0, len(texto) - 1),
        (texto.find("{"), texto.rfind("}")),
        (texto.find("["), texto.rfind("]")),
    ):
        if inicio < 0 or fin <= inicio:
            continue
        try:
            dato = json.loads(texto[inicio:fin + 1])
            break
        except json.JSONDecodeError:
            continue
    if dato is None:
        return None
    motivo = ""
    if isinstance(dato, dict):
        motivo = str(dato.get("motivo", ""))[:180]
        valores = dato.get("sintomas", dato.get("labels"))
    else:
        valores = dato
    if not isinstance(valores, list) or len(valores) != len(BDI_21):
        return None
    bits = []
    for v in valores:
        try:
            bits.append(1 if int(v) >= 1 else 0)
        except (TypeError, ValueError):
            return None
    return bits, motivo


def consultar_glm_sintomas(titulo, texto, reintentos=4, tope=2400):
    clave = os.environ.get("MISTRAL_API_KEY", "").strip()
    if not clave:
        raise RuntimeError("Falta MISTRAL_API_KEY en el entorno para preetiquetar síntomas con GLM.")
    cabeceras = {"Authorization": f"Bearer {clave}", "Content-Type": "application/json"}
    carga = {
        "model": MODELO_GLM,
        "temperature": 0.0,
        "max_tokens": tope,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SISTEMA_BDI21},
            {"role": "user", "content": f"TITLE: {titulo}\nBODY: {texto}"},
        ],
    }
    for intento in range(reintentos):
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=cabeceras, json=carga, timeout=120)
        except requests.RequestException:
            time.sleep(min(2 ** intento, 15))
            continue
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(min(2 ** intento + random.random(), 20))
            continue
        if r.status_code != 200:
            return None
        try:
            el = r.json()["choices"][0]
            res = parsear_sintomas_21(texto_de(el["message"]))
        except (KeyError, IndexError, TypeError, ValueError):
            res = None
        if res is not None:
            return res
        if el.get("finish_reason") == "length" and tope < 8000:
            return consultar_glm_sintomas(titulo, texto, reintentos=max(1, reintentos - 1), tope=tope * 2)
    return None


ejecutar_preetiquetado_bdi21(consultar_glm_sintomas, "glm", trabajadores=TRABAJADORES_SINTOMAS)


[glm] en caché BDI-21: 0 · pendientes: 180


BDI-21 (glm):   0%|          | 0/180 [00:00<?, ?post/s]

[glm] posts sin preetiqueta: 0


### Opción B · Preetiquetar 21 síntomas con StartLux local (`/v1/systemone`)

Usa el mismo protocolo `systemone` (`state.post` + preguntas `type: "noul"` y respuesta `answers[clave]["noul"]`) que en `bdi_clasificador.ipynb`. Asegúrate de tener el servidor levantado en `127.0.0.1:8889`.


In [43]:
STARTLUX_HOST = "127.0.0.1"
STARTLUX_PORT = 8889
MAX_CHARS_STARTLUX = 3000
UMBRAL_SINTOMA = 0.5
TRABAJADORES_STARTLUX = 8

PREGUNTAS_BDI21_SYSTEMONE = {
    f"s{n:02d}": {
        "type": "noul",
        "instructions": (
            f"Does the author explicitly describe that they themselves experience "
            f"the depression symptom '{en}' ({es})? "
            f"Gender dysphoria alone or mentions of other people do NOT count."
        ),
    }
    for n, en, es in BDI_21
}

_tl_startlux = threading.local()


def _conn_startlux():
    c = getattr(_tl_startlux, "c", None)
    if c is None:
        c = _tl_startlux.c = http.client.HTTPConnection(STARTLUX_HOST, STARTLUX_PORT, timeout=120)
    return c


def consultar_startlux_sintomas(titulo, texto, reintentos=5):
    cuerpo_texto = f"{titulo}\n\n{texto}".strip()[:MAX_CHARS_STARTLUX]
    body = json.dumps({
        "state": {"post": cuerpo_texto},
        "think_mode": "never",
        "questions": PREGUNTAS_BDI21_SYSTEMONE,
    })
    for intento in range(reintentos):
        try:
            c = _conn_startlux()
            c.request("POST", "/v1/systemone", body, {"Content-Type": "application/json"})
            r = c.getresponse()
            data = r.read()
            if r.status != 200:
                raise RuntimeError(f"HTTP {r.status}: {data[:200]!r}")
            ans = json.loads(data)["answers"]
            probs = [float(ans[f"s{n:02d}"]["noul"]) for n, _, _ in BDI_21]
            bits = [1 if p >= UMBRAL_SINTOMA else 0 for p in probs]
            top = sorted(
                [(i + 1, p) for i, p in enumerate(probs) if p >= UMBRAL_SINTOMA],
                key=lambda x: x[1],
                reverse=True,
            )[:5]
            motivo = (
                "startlux: " + ", ".join(f"s{num:02d}={p:.2f}" for num, p in top)
                if top
                else f"startlux: ninguno >= {UMBRAL_SINTOMA}"
            )
            return bits, motivo, {"probs": [round(p, 4) for p in probs]}
        except Exception:
            _tl_startlux.c = None
            if intento == reintentos - 1:
                return None
            time.sleep(0.5 * (2 ** intento))
    return None


ejecutar_preetiquetado_bdi21(
    consultar_startlux_sintomas,
    "startlux",
    trabajadores=TRABAJADORES_STARTLUX,
)


[startlux] en caché BDI-21: 180 · pendientes: 0


In [45]:
if not positivos.empty:
    cache_b21 = {
        str(r.get("id", "")).strip(): r
        for r in cargar_jsonl(PREETIQUETAS_BDI21_JSONL)
        if isinstance(r.get("bits"), list) and len(r["bits"]) == 21
    }
    previas = {}
    if ARCHIVO_BDI21.exists():
        df_prev = pd.read_csv(ARCHIVO_BDI21, encoding="utf-8-sig", dtype=str).fillna("")
        previas = {row["id"]: row for row in df_prev.to_dict("records") if row.get("id")}

    nuevas = []
    for row in positivos.fillna("").to_dict("records"):
        pid = str(row["id"])
        ant = previas.get(pid, {})
        reg = cache_b21.get(pid)
        revisado = str(ant.get("revisado", "")).strip()

        item = {
            "orden": row.get("orden", ""),
            "id": pid,
            "subreddit": row.get("subreddit", ""),
            "anio": row.get("anio", ""),
            "palabras": row.get("palabras", ""),
            "bdi_prob": row.get("bdi_prob", ""),
            "titulo": row.get("titulo", ""),
            "texto": row.get("texto", ""),
            "pre_modelo": ant.get("pre_modelo") or (reg.get("modelo", "") if reg else ""),
            "pre_sintomas": ant.get("pre_sintomas") or (reg.get("pre_sintomas", "") if reg else ""),
            "pre_motivo": ant.get("pre_motivo") or (reg.get("pre_motivo", "") if reg else ""),
            "gold_sintomas": ant.get("gold_sintomas", ""),
            "revisado": revisado,
        }
        bits = reg.get("bits") if reg else None
        for idx, col in enumerate(CAMPOS_S):
            if revisado in ("1", "9"):
                item[col] = ant.get(col, "")
            elif bits is not None:
                item[col] = str(bits[idx])
            else:
                item[col] = ant.get(col, "")
        nuevas.append(item)

    df_b21 = pd.DataFrame(nuevas, columns=COLUMNAS_BDI21)
    tmp_b21 = ARCHIVO_BDI21.with_suffix(".tmp")
    df_b21.to_csv(tmp_b21, index=False, encoding="utf-8-sig")
    os.replace(tmp_b21, ARCHIVO_BDI21)
    print(f"Escrito {ARCHIVO_BDI21}: {len(df_b21):,} filas · revisadas: {int(df_b21['revisado'].isin(['1', '9']).sum()):,}")


Escrito datos/gold_bdi21.csv: 180 filas · revisadas: 0


/tmp/ipykernel_2437/4202846427.py:13: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  for row in positivos.fillna("").to_dict("records"):


La revisión de los 21 síntomas se hace en terminal con:

```bash
cd "02_dataset_gold"
py revisar_sintomas.py        # en WSL: python3 revisar_sintomas.py
```

Las teclas permiten aceptar con `Enter` o `a`, alternar síntomas escribiendo sus números (`1 7 14`), agregar/quitar (`+14`, `-7`), limpiar (`0`), marcar dudoso (`9`), deshacer (`u`) o pausar (`q`). La última celda resume la frecuencia de los 21 síntomas ya revisados.


In [47]:
if ARCHIVO_BDI21.exists():
    res_b21 = pd.read_csv(ARCHIVO_BDI21, encoding="utf-8-sig", dtype=str).fillna("")
    revisados = res_b21[res_b21["revisado"] == "1"].copy()
    print(f"gold_bdi21.csv: {len(res_b21):,} filas · confirmadas (revisado=1): {len(revisados):,} · "
          f"dudosas (9): {int((res_b21['revisado'] == '9').sum()):,}")
    if not revisados.empty:
        conteos = []
        for n, en, es in BDI_21:
            col = f"s{n:02d}"
            c = int((pd.to_numeric(revisados[col], errors="coerce") == 1).sum())
            conteos.append({"num": n, "sintoma": f"{en} ({es})", "n": c, "pct": round(100 * c / len(revisados), 1)})
        display(pd.DataFrame(conteos))
else:
    print("Aún no existe datos/gold_bdi21.csv.")


gold_bdi21.csv: 180 filas · confirmadas (revisado=1): 180 · dudosas (9): 0


,num,sintoma,n,pct
0,1,Sadness (Tristeza),92,51.1
1,2,Pessimism (Pesimismo),44,24.4
2,3,Past Failure (Fracaso pasado),7,3.9
3,4,Loss of Pleasure (Perdida de placer),9,5.0
4,5,Guilty Feelings (Sentimientos de culpa),28,15.6
5,6,Punishment Feelings (Sentimientos de castigo),2,1.1
6,7,Self-Dislike (Rechazo hacia si mismo),43,23.9
7,8,Self-Criticalness (Autocritica),45,25.0
8,9,Suicidal Thoughts or Wishes (Pensamientos suic...,21,11.7
9,10,Crying (Llanto),33,18.3
